# sorted-computational-graph — worked example 3: Topological sort raises ValueError on a cyclic graph

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sorted-computational-graph`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A valid computation graph must be a directed acyclic graph (DAG). If a cycle exists — where following parent links eventually leads back to a node already on the DFS stack — the three-color DFS detects it when it re-encounters a node in the `temp` (gray) set. A ValueError is raised immediately, preventing infinite recursion.

## Worked solution

**Step 1 — Build a graph that has a back edge.** We manually insert a cycle: `c.recipe` references `d` as a parent, but `d.recipe` references `c`, creating a loop.

**Step 2 — Try to run topological_sort.** The DFS will visit c, then follow to d, then follow back to c — which is already in `temp`. This triggers the cycle detection.

**Step 3 — Catch the ValueError.** We use a try/except to confirm the error is raised and print the message.

**Step 4 — Show that non-cyclic graphs still work.** A valid DAG of the same nodes (without the back edge) sorts correctly.

In [ ]:
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name):
        self.name = name
        self.recipe = None
    def __repr__(self):
        return f'T({self.name})'

def topological_sort(node, get_children):
    result = []
    perm = set()
    temp = set()
    def visit(cur):
        cid = id(cur)
        if cid in perm:
            return
        if cid in temp:
            raise ValueError(f'Cycle detected at {cur!r}')
        temp.add(cid)
        for child in get_children(cur):
            visit(child)
        temp.discard(cid)
        perm.add(cid)
        result.append(cur)
    visit(node)
    return result

def get_parents(n):
    if n.recipe is None:
        return []
    return list(n.recipe.parents.values())

# Build a cycle: x -> y -> x
x = FakeTensor('x')
y = FakeTensor('y')
x.recipe = FakeRecipe([y])   # x depends on y
y.recipe = FakeRecipe([x])   # y depends on x -- CYCLE!

try:
    topological_sort(x, get_parents)
    print('ERROR: should have raised ValueError')
except ValueError as e:
    print(f'Correctly caught cycle: {e}')

# Valid DAG for comparison
a = FakeTensor('a')
b = FakeTensor('b')
b.recipe = FakeRecipe([a])
result = topological_sort(b, get_parents)[::-1]
print('Valid DAG bwd order:', [n.name for n in result])  # [b, a]